# Film Boiling Testcase
This is a testcase for evaporation.  
A thin vapor film is situated above a heated wall.  
The flat liquid-vapor interface is disturbed according to the largest rayleigh-taylor instability wavelength.  
As the liquid evaporated a vapor bubble is forming and eventually emerging from the film.  
references for this testcase (in some points they possess some discrepancies)  
`10.1016/j.jcp.2006.07.035` (G)  
`10.1006/jcpn.200.6481` (WW)     

Note the following:
* We take the case with $\Delta T = 5K$
* We use the value for $h_v$ given in (WW)

In [1]:
#r "..\..\binaries\BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using System.IO;
using System.Data;
using System.Globalization;
using System.Threading;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSFE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
using BoSSS.Foundation.Grid.RefElements;
using BoSSS.Platform.LinAlg;
using Microsoft.AspNetCore.Html;
Init();

In [2]:
string proj = "XNSFE-FilmBoiling-v2";
BoSSSshell.WorkflowMgm.Init(proj);
BoSSSshell.WorkflowMgm.SetNameBasedSessionJobControlCorrelation();

Project name is set to 'XNSFE-FilmBoiling-v2'.
Opening existing database '\\dc1\userspace\rieckmann\cluster\XNSFE-FilmBoiling-v2'.


In [3]:
var sessions = BoSSSshell.WorkflowMgm.DefaultDatabase.Sessions;
sessions

#0: XNSFE-FilmBoiling-v2	XNSFE-FilmBoiling-v2_H2_AMR4_P2_Convergence	01/02/2023 13:12:07	b9633827...
#1: XNSFE-FilmBoiling-v2	XNSFE-FilmBoiling-v2_H2_AMR3_P3_Convergence	01/02/2023 13:12:25	0cc85221...
#2: XNSFE-FilmBoiling-v2	XNSFE-FilmBoiling-v2_H2_AMR2_P3_Convergence	01/02/2023 13:12:16	0d86e3fd...
#3: XNSFE-FilmBoiling-v2	XNSFE-FilmBoiling-v2_H2_AMR3_P2_Convergence	01/02/2023 13:11:59	7074b08a...
#4: XNSFE-FilmBoiling-v2	XNSFE-FilmBoiling-v2_H2_AMR2_P2_Convergence	01/02/2023 13:11:54	6cf135ae...
#5: XNSFE-FilmBoiling-v2	XNSFE-FilmBoiling-v2_H2_AMR4_P3_Convergence*	01/02/2023 13:12:33	2347396e...


## Grid and Boundary/Initial Value configuration

In [4]:
static class Constants{
    // careful order of declaration matters!!!
    // material parameters
    public static double rho_A = 200.0;
    public static double rho_B = 5.0;

    public static double mu_A = 0.1;
    public static double mu_B = 0.005;

    public static double Sigma = 0.1;

    public static double c_A = 400.0;
    public static double c_B = 200.0;

    public static double k_A = 40.0;
    public static double k_B = 1.0;

    public static double hVap  = 10000;
    public static double T_sat = 0; // 500K, shifted
    public static double T_wall = T_sat + 5.0;

    public static double alpha_A = k_A / (c_A*rho_A);
    public static double alpha_B = k_B / (c_B*rho_B);
    public static double eps = 1.0-rho_B/rho_A;

    public static double g = 9.81;

    public static double lambda = 2 * Math.PI * Math.Sqrt((3*Sigma)/(g*Math.Abs(rho_A-rho_B)));
    public static double L = lambda / 2.0;

    // capillary timestep , for finest res + highest degree, use this, for comparability?!, Is very small 1e-7 => 1e5 - 1e6 timesteps necessary => artificial surface tension?!
    public static Func<int, int, double> dt = (res, p) => 0.5 * Math.Sqrt((rho_A + rho_B) * Math.Pow(L / ((double)res * ((double)p + 1)), 3) / (2 * Math.PI * Math.Abs(Sigma)));
}

In [5]:
Constants.L

0.03934220442504576

## Begin Postprocessing

In [6]:
//var sessions = BoSSSshell.WorkflowMgm.Sessions.Where(s => s.SuccessfulTermination);
//var sessions = BoSSSshell.WorkflowMgm.Sessions.Where(s => Convert.ToDouble(s.KeysAndQueries["DGdegree:Velocity*"]) == 3 && Convert.ToString(s.RestartedFrom) != new Guid().ToString());
//var sessions = BoSSSshell.WorkflowMgm.Sessions.Where(s => Convert.ToDouble(s.KeysAndQueries["DGdegree:Velocity*"]) == 3);
var sessions = BoSSSshell.WorkflowMgm.Sessions.ToList();
sessions = sessions.Where(s => s.Name.Contains("v2")).ToList();
sessions = sessions.OrderByDescending(s => s.KeysAndQueries["id:AMR"]).ToList();
sessions

#0: XNSFE-FilmBoiling-v2	XNSFE-FilmBoiling-v2_H2_AMR4_P2_Convergence	01/02/2023 13:12:07	b9633827...
#1: XNSFE-FilmBoiling-v2	XNSFE-FilmBoiling-v2_H2_AMR4_P3_Convergence*	01/02/2023 13:12:33	2347396e...
#2: XNSFE-FilmBoiling-v2	XNSFE-FilmBoiling-v2_H2_AMR3_P3_Convergence	01/02/2023 13:12:25	0cc85221...
#3: XNSFE-FilmBoiling-v2	XNSFE-FilmBoiling-v2_H2_AMR3_P2_Convergence	01/02/2023 13:11:59	7074b08a...
#4: XNSFE-FilmBoiling-v2	XNSFE-FilmBoiling-v2_H2_AMR2_P3_Convergence	01/02/2023 13:12:16	0d86e3fd...
#5: XNSFE-FilmBoiling-v2	XNSFE-FilmBoiling-v2_H2_AMR2_P2_Convergence	01/02/2023 13:11:54	6cf135ae...


Lets sort the sessions by degree

In [7]:
SortedDictionary<int, List<ISessionInfo>> groupedSessions = new SortedDictionary<int, List<ISessionInfo>>(sessions.GroupBy(s => Convert.ToInt32(s.KeysAndQueries["id:Degree"])).ToDictionary(g => g.Key, g => g.ToList()));

Load Log Data

In [8]:
var data = groupedSessions.ToDictionary(kvp => kvp.Key, kvp => kvp.Value.ReadLogDataForXNSE("Massflux"));

Element at 0: time vs mass-liq
Element at 1: time vs mass-vap
Element at 2: time vs mass-total
Element at 3: time vs masschange-evap
Element at 4: time vs masschange-vapor
Element at 5: time vs masschange-liquid
Element at 6: time vs masschange-total
Element at 7: time vs interface length
Element at 0: time vs mass-liq
Element at 1: time vs mass-vap
Element at 2: time vs mass-total
Element at 3: time vs masschange-evap
Element at 4: time vs masschange-vapor
Element at 5: time vs masschange-liquid
Element at 6: time vs masschange-total
Element at 7: time vs interface length


### TODO: Animation of interface and temperature isocontours

In [9]:
// var sess2plot = sessions.First();//groupedSessions[groupedSessions.Keys.Max()].OrderBy(s => s.KeysAndQueries["Grid:hMax"]).First();// pick session with finest grid and highest degree
// string path2plot = Path.GetFullPath(Path.Combine("./Plots", sess2plot.Name));
// sess2plot.Export().To(path2plot).WithSupersampling(2).WithShadowFields().Do(true);

In [10]:
# #!pwsh
# #!share --from csharp path2plot
# pvpython ./Plots/filmboiling.py $path2plot

In [11]:
// display(new HtmlString(String.Format("<video width=\"542\" height=\"796\" controls><source src=\"{0}\" type=\'video/ogg; codecs=\"theora\"\'></video>",Path.Combine("./Plots", sess2plot.Name, "filmboiling.ogv") )))

Add more postprocessing, what else to look at?

In [12]:
// var ts = new List<ITimestepInfo>();
// sessions.ForEach(s => ts.AddRange(s.Timesteps));
// ts = ts.Where(t => t.TimeStepNumber.Length == 1).GroupBy(t => t.TimeStepNumber.MajorNumber).Select(t => t.First()).ToList();
// string path2plot = Path.GetFullPath(Path.Combine("./Plots", "FilmBoilingRestart"));
// ts.Export().To(path2plot).WithSupersampling(2).WithShadowFields().Do(true);

# Plot Contour

In [13]:
using BoSSS.Foundation.Quadrature;
using BoSSS.Foundation.XDG;

In [14]:
public static int order = 0; // formerly 8, controls somewhat number of points

public static (double[], double[]) GetInterfaceContour(ITimestepInfo ts){
    var f = (LevelSet)ts.Fields.Single(f => f.Identification == "Phi");
    var LsTrk = new LevelSetTracker((GridData)f.GridDat, XQuadFactoryHelper.MomentFittingVariants.Saye, 1, new[] {"A", "B"}, f);
    var SchemeHelper = LsTrk.GetXDGSpaceMetrics(LsTrk.SpeciesIdS.ToArray(), order, 1).XQuadSchemeHelper;
    var lqs = SchemeHelper.GetLevelSetquadScheme(0, LsTrk.Regions.GetCutCellMask());
    
    List<double> X = new List<double>();
    List<double> Y = new List<double>();
    
    CellQuadrature.GetQuadrature(new int[] { 1 }, LsTrk.GridDat,
        lqs.Compile(LsTrk.GridDat, order),
        delegate (int i0, int Length, QuadRule QR, MultidimensionalArray EvalResult) {
            MultidimensionalArray GlobalNodes = MultidimensionalArray.Create(Length, QR.NoOfNodes, 2);
            for(int i = 0; i < Length; i++){
                LsTrk.GridDat.TransformLocal2Global(QR.Nodes, i0+i, 1, GlobalNodes, 0);          
                int K = QR.NoOfNodes;
                for(int k = 0; k < K; k++){
                    X.Add(GlobalNodes[i, k, 0]);
                    Y.Add(GlobalNodes[i, k, 1]);
                }
            }
        },
        delegate (int i0, int Length, MultidimensionalArray ResultsOfIntegration) {
        }
    ).Execute();

    return (X.ToArray(), Y.ToArray());
}

public static (string, double[], double[]) ReadReferenceCSV(string file){
    List<double> X = new List<double>();
    List<double> Y = new List<double>();
    string name;
    using(var reader = new StreamReader(file))
    {
        var line = reader.ReadLine();
        var values = line.Split(';');    
        name = values[1];

        while (!reader.EndOfStream)
        {
            line = reader.ReadLine();
            values = line.Split(';');

            X.Add(Convert.ToDouble(values[0].Replace(',','.')));
            Y.Add(Convert.ToDouble(values[1].Replace(',','.')));
        }
    }
    return (name, X.ToArray(), Y.ToArray());
}

In [15]:
//var sess = sessions.Pick(0);
//var ts = sess.Timesteps.Where(t => t.TimeStepNumber.Length == 1).SingleOrDefault(t => t.TimeStepNumber.MajorNumber == 0);
//
//int[] Steps = {900, 920, 940, 960, 980};
//string[] Formats = {"k+","r+","k+","r+", "k+"};
//
//Dictionary<double, (double[], double[])> InterfaceEvo = new Dictionary<double, (double[], double[])>();
//Steps.ForEach(i => {
//    var ts = sess.Timesteps.Where(t => t.TimeStepNumber.Length == 1).SingleOrDefault(t => t.TimeStepNumber.MajorNumber == i);
//    InterfaceEvo.Add(ts.PhysicalTime, GetInterfaceContour(ts));
//});
//
//List<double>[] X = InterfaceEvo.Values.Select(v => v.Item1.ToList()).ToArray();
//List<double>[] Y = InterfaceEvo.Values.Select(v => v.Item2.ToList()).ToArray();
//string[] titles = InterfaceEvo.Keys.Select(v => "t = " + v).ToArray();
//
//Plot(X, Y, titles, Formats)

In [16]:
foreach(var kvp in groupedSessions){

    string[] Formats = {"g+","r+","b+","ko"};

    var Reference = ReadReferenceCSV("GibouFilmboiling.csv");

    Dictionary<string, (double[], double[])> InterfaceEvo = new Dictionary<string, (double[], double[])>();
    kvp.Value.ForEach(i => {
        //int k = Math.Min(780, i.Timesteps.Count - 1);
        int k = Math.Min(870, i.Timesteps.Count - 1);
        //int k = Math.Min(100, i.Timesteps.Count - 1);

        while(k < i.Timesteps.Count - 1){
            double time = i.Timesteps.ElementAt(k).PhysicalTime;
            if( time > 0.425) break;
            k++;
        }
        var ts = i.Timesteps.ElementAt(k);
        String.Format("{1} : t={0}", ts.PhysicalTime, ts.Session.Name).Display();

        // string id = i.Name;
        string id = String.Format("p={0}, h={1}, amr={2}", i.KeysAndQueries["id:Degree"], i.KeysAndQueries["id:Res"], i.KeysAndQueries["id:AMR"]);
        InterfaceEvo.Add(id, GetInterfaceContour(ts));
    });

    List<double>[] X = InterfaceEvo.Values.Select(v => v.Item1.ToList()).Cat(Reference.Item2.ToList()).ToArray();
    List<double>[] Y = InterfaceEvo.Values.Select(v => v.Item2.ToList()).Cat(Reference.Item3.ToList()).ToArray();
    string[] titles = InterfaceEvo.Keys.Select(v => v).Cat(Reference.Item1).ToArray();

    

    Plot2Ddata plot = new Plot2Ddata();
    for(int i = 0; i < X.Length; i++){
        plot.AddDataGroup(titles[i], X[i], Y[i], Formats[i]);
    }
    var gp = plot.ToGnuplot();
    gp.WriteScriptAndData("./Figures/gnuplot/data/Filmboiling_p"+kvp.Key); // change a few settings by hand, see readme!

    //Plot(X, Y, titles, Formats);
    HtmlString pp = new HtmlString(
        "<div>" +
        "   <div style='float:left;'>" +
            plot.PlotNow().ToString() +
        "   </div>" +
        "</div>");
    display(pp);
}

XNSFE-FilmBoiling-v2_H2_AMR4_P2_Convergence : t=0.42534517913424585

XNSFE-FilmBoiling-v2_H2_AMR3_P2_Convergence : t=0.42534517913424585

XNSFE-FilmBoiling-v2_H2_AMR2_P2_Convergence : t=0.42534517913424585

Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe
Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe


<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 0.005 
 
 
 
 
 0.01 
 
 
 
 
 0.015 
 
 
 
 
 0.02 
 
 
 
 
 0.025 
 
 
 
 
 0.03 
 
 
 
 
 0.035 
 
 
 
 
 0.04 
 
 
 
 
 -0.04 
 
 
 
 
 -0.03 
 
 
 
 
 -0.02 
 
 
 
 
 -0.01 
 
 
 
 
 0 
 
 
 
 
 0.01 
 
 
 
 
 0.02 
 
 
 
 
 0.03 
 
 
 
 
 0.04 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 p=2, h=2, amr=4 
 
 
 p=2, h=2, amr=4 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 p=2, h=2, amr=3 
 
 
 p=2, h=2, amr=3 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 p=2, h=2, amr=2 
 
 
 p=2, h=2, amr=2 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 Gibou 280x280 
 
 
 Gibou 280x280

XNSFE-FilmBoiling-v2_H2_AMR4_P3_Convergence : t=0.42534517913424585

XNSFE-FilmBoiling-v2_H2_AMR3_P3_Convergence : t=0.42534517913424585

XNSFE-FilmBoiling-v2_H2_AMR2_P3_Convergence : t=0.42534517913424585

Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe
Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe


<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 0.005 
 
 
 
 
 0.01 
 
 
 
 
 0.015 
 
 
 
 
 0.02 
 
 
 
 
 0.025 
 
 
 
 
 0.03 
 
 
 
 
 0.035 
 
 
 
 
 0.04 
 
 
 
 
 -0.04 
 
 
 
 
 -0.03 
 
 
 
 
 -0.02 
 
 
 
 
 -0.01 
 
 
 
 
 0 
 
 
 
 
 0.01 
 
 
 
 
 0.02 
 
 
 
 
 0.03 
 
 
 
 
 0.04 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 p=3, h=2, amr=4 
 
 
 p=3, h=2, amr=4 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 p=3, h=2, amr=3 
 
 
 p=3, h=2, amr=3 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 p=3, h=2, amr=2 
 
 
 p=3, h=2, amr=2 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 Gibou 280x280 
 
 
 Gibou 280x280

In [19]:
public static Plot2Ddata PlotInterfaceEvolution(ISessionInfo sess, int levels, bool seperate = false){
    return PlotInterfaceEvolution(sess.Timesteps, levels, seperate);
}

public static Plot2Ddata PlotInterfaceEvolution(IEnumerable<ITimestepInfo> tsteps, int levels, bool seperate = false){
    tsteps = tsteps.Where(ts => ts.TimeStepNumber.Numbers.Length == 1);
    int[] inc = GenericBlas.Linspace(0, tsteps.Count() - 1, levels).Select(d => (int)d).ToArray();

    string[] Formats = levels.ForLoop(l => l % 2 == 0 ? "k+" : "kx");

    Dictionary<string, (double[], double[])> InterfaceEvo = new Dictionary<string, (double[], double[])>();

    tsteps.Pick(inc).ForEach(ts => {
        String.Format("{1} : t={0}", ts.PhysicalTime, ts.TimeStepNumber).Display();
        InterfaceEvo.Add(ts.TimeStepNumber.MajorNumber.ToString(), GetInterfaceContour(ts));
    });

    List<double>[] X = InterfaceEvo.Values.Select(v => v.Item1.ToList()).ToArray();
    List<double>[] Y = InterfaceEvo.Values.Select(v => v.Item2.ToList()).ToArray();
    string[] titles = InterfaceEvo.Keys.Select(v => v).ToArray();

    var plt = new Plot2Ddata();
    for(int i = 0; i < levels; i++){
        plt.AddDataGroup(titles.ElementAt(i), X.ElementAt(i), Y.ElementAt(i), Formats.ElementAt(i));
    }

    if(seperate){
        for(int i = 0; i < levels; i++){
            Plot(X.ElementAt(i), Y.ElementAt(i), titles.ElementAt(i), Formats.ElementAt(i)).Display();
        }
    }
    else{
        Plot(X, Y, titles, Formats).Display();
    }
    return plt;
}


In [27]:
sessions

#0: XNSFE-FilmBoiling-v2	XNSFE-FilmBoiling-v2_H2_AMR4_P2_Convergence	01/02/2023 13:12:07	b9633827...
#1: XNSFE-FilmBoiling-v2	XNSFE-FilmBoiling-v2_H2_AMR4_P3_Convergence*	01/02/2023 13:12:33	2347396e...
#2: XNSFE-FilmBoiling-v2	XNSFE-FilmBoiling-v2_H2_AMR3_P3_Convergence	01/02/2023 13:12:25	0cc85221...
#3: XNSFE-FilmBoiling-v2	XNSFE-FilmBoiling-v2_H2_AMR3_P2_Convergence	01/02/2023 13:11:59	7074b08a...
#4: XNSFE-FilmBoiling-v2	XNSFE-FilmBoiling-v2_H2_AMR2_P3_Convergence	01/02/2023 13:12:16	0d86e3fd...
#5: XNSFE-FilmBoiling-v2	XNSFE-FilmBoiling-v2_H2_AMR2_P2_Convergence	01/02/2023 13:11:54	6cf135ae...


In [21]:
PlotInterfaceEvolution(sessions.Pick(0), 4, true);

0 : t=0

3420 : t=0.16643941792207886

6850 : t=0.33336550080887073

10275 : t=0.5000482512133203

Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe


...

<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 0.0052 
 
 
 
 
 0.0054 
 
 
 
 
 0.0056 
 
 
 
 
 0.0058 
 
 
 
 
 0.006 
 
 
 
 
 0.0062 
 
 
 
 
 0.0064 
 
 
 
 
 0.0066 
 
 
 
 
 0.0068 
 
 
 
 
 0.007 
 
 
 
 
 0.0072 
 
 
 
 
 0.0074 
 
 
 
 
 -0.04 
 
 
 
 
 -0.03 
 
 
 
 
 -0.02 
 
 
 
 
 -0.01 
 
 
 
 
 0 
 
 
 
 
 0.01 
 
 
 
 
 0.02 
 
 
 
 
 0.03 
 
 
 
 
 0.04 
 
 
 
 
 
 
 
 
 3420 
 
 
 3420

Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe


<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 0.002 
 
 
 
 
 0.004 
 
 
 
 
 0.006 
 
 
 
 
 0.008 
 
 
 
 
 0.01 
 
 
 
 
 0.012 
 
 
 
 
 0.014 
 
 
 
 
 0.016 
 
 
 
 
 0.018 
 
 
 
 
 -0.04 
 
 
 
 
 -0.03 
 
 
 
 
 -0.02 
 
 
 
 
 -0.01 
 
 
 
 
 0 
 
 
 
 
 0.01 
 
 
 
 
 0.02 
 
 
 
 
 0.03 
 
 
 
 
 0.04 
 
 
 
 
 
 
 
 
 6850 
 
 
 6850

Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe


<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 0.005 
 
 
 
 
 0.01 
 
 
 
 
 0.015 
 
 
 
 
 0.02 
 
 
 
 
 0.025 
 
 
 
 
 0.03 
 
 
 
 
 0.035 
 
 
 
 
 0.04 
 
 
 
 
 0.045 
 
 
 
 
 0.05 
 
 
 
 
 0.055 
 
 
 
 
 -0.04 
 
 
 
 
 -0.03 
 
 
 
 
 -0.02 
 
 
 
 
 -0.01 
 
 
 
 
 0 
 
 
 
 
 0.01 
 
 
 
 
 0.02 
 
 
 
 
 0.03 
 
 
 
 
 0.04 
 
 
 
 
 
 
 
 
 10275 
 
 
 10275

In [22]:
var plt0 = PlotInterfaceEvolution(sessions.Pick(5), 4, true);
var plt1 = PlotInterfaceEvolution(sessions.Pick(0), 4, true);

0 : t=0

3420 : t=0.16643941792207886

6850 : t=0.33336550080887073

10275 : t=0.5000482512133203

Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe


...

<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 0.0052 
 
 
 
 
 0.0054 
 
 
 
 
 0.0056 
 
 
 
 
 0.0058 
 
 
 
 
 0.006 
 
 
 
 
 0.0062 
 
 
 
 
 0.0064 
 
 
 
 
 0.0066 
 
 
 
 
 0.0068 
 
 
 
 
 0.007 
 
 
 
 
 0.0072 
 
 
 
 
 0.0074 
 
 
 
 
 -0.04 
 
 
 
 
 -0.03 
 
 
 
 
 -0.02 
 
 
 
 
 -0.01 
 
 
 
 
 0 
 
 
 
 
 0.01 
 
 
 
 
 0.02 
 
 
 
 
 0.03 
 
 
 
 
 0.04 
 
 
 
 
 
 
 
 
 3420 
 
 
 3420

Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe


<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 0.002 
 
 
 
 
 0.004 
 
 
 
 
 0.006 
 
 
 
 
 0.008 
 
 
 
 
 0.01 
 
 
 
 
 0.012 
 
 
 
 
 0.014 
 
 
 
 
 0.016 
 
 
 
 
 0.018 
 
 
 
 
 -0.04 
 
 
 
 
 -0.03 
 
 
 
 
 -0.02 
 
 
 
 
 -0.01 
 
 
 
 
 0 
 
 
 
 
 0.01 
 
 
 
 
 0.02 
 
 
 
 
 0.03 
 
 
 
 
 0.04 
 
 
 
 
 
 
 
 
 6850 
 
 
 6850

Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe


<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 0.005 
 
 
 
 
 0.01 
 
 
 
 
 0.015 
 
 
 
 
 0.02 
 
 
 
 
 0.025 
 
 
 
 
 0.03 
 
 
 
 
 0.035 
 
 
 
 
 0.04 
 
 
 
 
 0.045 
 
 
 
 
 0.05 
 
 
 
 
 0.055 
 
 
 
 
 -0.04 
 
 
 
 
 -0.03 
 
 
 
 
 -0.02 
 
 
 
 
 -0.01 
 
 
 
 
 0 
 
 
 
 
 0.01 
 
 
 
 
 0.02 
 
 
 
 
 0.03 
 
 
 
 
 0.04 
 
 
 
 
 
 
 
 
 10275 
 
 
 10275

In [23]:
    var gp = plt0.ToGnuplot();
    gp.WriteScriptAndData("./Figures/gnuplot/data/Filmboiling_p2_coarse"); // change a few settings by hand, see readme!
    gp = plt1.ToGnuplot();
    gp.WriteScriptAndData("./Figures/gnuplot/data/Filmboiling_p2_fine"); // change a few settings by hand, see readme!

Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe
Using gnuplot: C:\Program Files (x86)\FDY\BoSSS\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe


In [33]:
public static void ExportSession(ISessionInfo sess, int cnt){
    ExportSession(sess.Timesteps, cnt, sess.Name);
}

public static void ExportSession(IEnumerable<ITimestepInfo> timesteps, int cnt, string name){
    var Timesteps = timesteps.Where(ts => ts.TimeStepNumber.Numbers.Length == 1);
    int[] steps = GenericBlas.Linspace(0, Timesteps.Count() - 1, cnt).Select(d => (int)d).ToArray();
    var tsteps = Timesteps.Pick(steps);
    string path2plot = Path.GetFullPath(Path.Combine("./Plots", name));
    tsteps.Export().To(path2plot).WithSupersampling(2).WithShadowFields().Do(false);
}

In [34]:
ExportSession(sessions.Pick(0), 300);

Starting export process... Data will be written to the directory: d:\rieckmann\BoSSS\BoSSS-Repos\BoSSS-Master\internal\src\private-rckmnn\notebooks\XNSFE-Paper\2D-Filmboiling\Plots\XNSFE-FilmBoiling-v2_H2_AMR4_P2_Convergence
